# TAPS-MCC — Edge IDS Experiment (Edge-IIoTset) — v2
*v2: fixes the "0" vs "0.0" label leakage, caps SMOTE, bounds tree depth, stabilises LightGBM, adds a leakage check.*
This notebook trains the lightweight intrusion detection module used for **behavioural trust** in TAPS-MCC (Eq. 2: BT = 1 − anomaly score).

**How to run:** `Runtime → Run all`. It takes about 15–30 minutes on a free Colab CPU.
At the end, a file `TAPS_IDS_results.zip` downloads automatically — upload it to Claude.

## 1. Settings

In [ ]:
# ---- You can change these ----
SAMPLE_ROWS = 300_000   # rows used (stratified sample) — keeps free Colab within RAM/time
TOP_K       = 30        # features kept after mutual-information selection
RUN_CNN     = True      # 1D-CNN (set False if the session is slow)
SEED        = 42
CSV_PATH    = None      # leave None to download from Kaggle automatically

## 2. Install and import

In [ ]:
!pip -q install kagglehub imbalanced-learn xgboost lightgbm
import os, glob, time, json, pickle, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
np.random.seed(SEED)
os.makedirs('results', exist_ok=True)

## 3. Load Edge-IIoTset (Ferrag et al., IEEE Access 2022)

In [ ]:
if CSV_PATH is None:
    import kagglehub
    root = kagglehub.dataset_download('mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot')
    hits = glob.glob(os.path.join(root, '**', 'DNN-EdgeIIoT-dataset.csv'), recursive=True)
    assert hits, 'DNN-EdgeIIoT-dataset.csv not found — upload it and set CSV_PATH'
    CSV_PATH = hits[0]
print('Reading', CSV_PATH)
df = pd.read_csv(CSV_PATH, low_memory=False)
print('Full dataset:', df.shape)
print(df['Attack_type'].value_counts())

## 4. Pre-processing (following the dataset authors' recommendations)

In [ ]:
DROP = ['frame.time','ip.src_host','ip.dst_host','arp.src.proto_ipv4','arp.dst.proto_ipv4',
        'http.file_data','http.request.full_uri','icmp.transmit_timestamp','http.request.uri.query',
        'tcp.options','tcp.payload','tcp.srcport','tcp.dstport','udp.port','mqtt.msg']
df = df.drop(columns=[c for c in DROP if c in df.columns])
df = df.dropna().drop_duplicates()

# stratified sample to fit free Colab
if SAMPLE_ROWS and len(df) > SAMPLE_ROWS:
    df, _ = train_test_split(df, train_size=SAMPLE_ROWS, stratify=df['Attack_type'], random_state=SEED)

y_bin = df['Attack_label'].astype(int).values
le = LabelEncoder(); y_mul = le.fit_transform(df['Attack_type'].astype(str))
X = df.drop(columns=['Attack_label','Attack_type'])

CAT = ['http.request.method','http.referer','http.request.version','dns.qry.name.len',
       'mqtt.conack.flags','mqtt.protoname','mqtt.topic']
CAT = [c for c in CAT if c in X.columns]
# v2: the dataset stores the same value as '0' in some files and '0.0' in others,
# which leaks the label. Normalise numeric-looking strings before encoding.
for c in CAT:
    X[c] = X[c].astype(str).str.strip().map(lambda v: v[:-2] if v.endswith('.0') and v[:-2].lstrip('-').isdigit() else v)
X = pd.get_dummies(X, columns=CAT)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype('float32')
print('After cleaning:', X.shape, '| classes:', list(le.classes_))

## 5. Train/test split and mutual-information feature selection

In [ ]:
X_tr, X_te, yb_tr, yb_te, ym_tr, ym_te = train_test_split(
    X, y_bin, y_mul, test_size=0.2, stratify=y_mul, random_state=SEED)

idx = np.random.RandomState(SEED).choice(len(X_tr), min(50_000, len(X_tr)), replace=False)
mi = mutual_info_classif(X_tr.iloc[idx], ym_tr[idx], random_state=SEED)
feat = X_tr.columns[np.argsort(mi)[::-1][:TOP_K]].tolist()
pd.Series(mi, index=X_tr.columns).sort_values(ascending=False).head(TOP_K).to_csv('results/selected_features.csv')
X_tr, X_te = X_tr[feat], X_te[feat]
print('Selected', len(feat), 'features')

## 6. Helper: train, time and score a model

In [ ]:
ROWS = []
def size_mb(model):
    if hasattr(model, 'm'):                      # Keras wrapper: float32 weights
        return model.m.count_params() * 4 / 1e6
    return len(pickle.dumps(model)) / 1e6

def evaluate(name, task, model, Xtr, ytr, Xte, yte, labels, fit=True):
    t0 = time.perf_counter()
    if fit: model.fit(Xtr, ytr)
    train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = model.predict(Xte); inf = time.perf_counter() - t0
    pred = np.asarray(pred).ravel().astype(int)
    acc = accuracy_score(yte, pred)
    p, r, f, _ = precision_recall_fscore_support(yte, pred, average='macro', zero_division=0)
    _, _, fw, _ = precision_recall_fscore_support(yte, pred, average='weighted', zero_division=0)
    cm = confusion_matrix(yte, pred)
    if task == 'binary':
        tn, fp, fn, tp = cm.ravel(); fpr = fp / (fp + tn)
    else:
        fpr = np.mean([(cm[:, k].sum() - cm[k, k]) / (cm.sum() - cm[k].sum()) for k in range(len(cm))])
    ROWS.append(dict(task=task, model=name, accuracy=acc, precision_macro=p, recall_macro=r,
                     f1_macro=f, f1_weighted=fw, fpr=fpr, train_s=train_s,
                     inference_us_per_flow=inf / len(Xte) * 1e6,
                     model_mb=size_mb(model) if fit else np.nan))
    fig, ax = plt.subplots(figsize=(7, 6) if task == 'multiclass' else (4, 4))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=90, colorbar=False, values_format='d')
    ax.set_title(f'{name} ({task})'); fig.tight_layout()
    fig.savefig(f'results/cm_{task}_{name}.png', dpi=200); plt.close(fig)
    print(f'{task:10s} {name:9s} acc={acc:.4f}  F1(macro)={f:.4f}  FPR={fpr:.4f}')
    return model

## 6b. Leakage check — can a single feature separate normal from attack?

In [ ]:
from sklearn.tree import DecisionTreeClassifier
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(X_tr, yb_tr)
stump_acc = accuracy_score(yb_te, stump.predict(X_te))
top = feat[int(np.argmax(stump.feature_importances_))]
leak = dict(single_feature_accuracy=round(float(stump_acc), 4), feature=top)
json.dump(leak, open('results/leakage_check.json', 'w'), indent=2)
print(leak, '-> WARNING: near-perfect single-feature split' if stump_acc > 0.99 else '-> OK')

## 7. SMOTE on the training set only (multiclass), then train models

In [ ]:
# v2: raise minority classes to at most 20,000 samples (no blanket balancing to the majority size)
cnt = pd.Series(ym_tr).value_counts()
target = min(20_000, int(cnt.max()))
sm = SMOTE(random_state=SEED, k_neighbors=3,
           sampling_strategy={int(k): max(int(v), target) for k, v in cnt.items()})
Xs_tr, ys_tr = sm.fit_resample(X_tr, ym_tr)
print('Multiclass training set after SMOTE:', Xs_tr.shape)

BIN_LABELS, MUL_LABELS = ['Normal', 'Attack'], list(le.classes_)
nc = len(MUL_LABELS)
models = {
 'RF':       lambda: RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
 'XGBoost':  lambda: XGBClassifier(n_estimators=200, max_depth=8, tree_method='hist', n_jobs=-1, random_state=SEED),
 'LightGBM': lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20, random_state=SEED, verbose=-1),
}
BIN_MODELS = {}
for name, make in models.items():
    BIN_MODELS[name] = evaluate(name, 'binary', make(), X_tr, yb_tr, X_te, yb_te, BIN_LABELS)
    evaluate(name, 'multiclass', make(), Xs_tr, ys_tr, X_te, ym_te, MUL_LABELS)

## 8. Lightweight 1D-CNN

In [ ]:
if RUN_CNN:
    import tensorflow as tf
    tf.random.set_seed(SEED)
    sc = StandardScaler().fit(Xs_tr)
    def cnn(n_out):
        m = tf.keras.Sequential([
            tf.keras.layers.Input((len(feat), 1)),
            tf.keras.layers.Conv1D(32, 3, activation='relu'),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Conv1D(64, 3, activation='relu'),
            tf.keras.layers.GlobalAveragePooling1D(),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(n_out, activation='softmax')])
        m.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
        return m
    class KerasWrap:
        def __init__(self, m): self.m = m
        def fit(self, X, y):
            self.m.fit(sc.transform(X)[..., None], y, epochs=5, batch_size=512, verbose=0, validation_split=0.1); return self
        def predict(self, X):
            return self.m.predict(sc.transform(X)[..., None], batch_size=2048, verbose=0).argmax(1)
    evaluate('1D-CNN', 'binary', KerasWrap(cnn(2)), X_tr, yb_tr, X_te, yb_te, BIN_LABELS)
    evaluate('1D-CNN', 'multiclass', KerasWrap(cnn(nc)), Xs_tr, ys_tr, X_te, ym_te, MUL_LABELS)

## 9. Behavioural trust: BT = 1 − anomaly score (TAPS-MCC Eq. 2)

In [ ]:
best = max([r for r in ROWS if r['task'] == 'binary' and r['model'] in BIN_MODELS], key=lambda r: r['f1_macro'])['model']
score = BIN_MODELS[best].predict_proba(X_te)[:, 1]      # s_i = P(attack)
BT = 1 - score
bt = pd.DataFrame({'Attack_type': le.inverse_transform(ym_te), 'BT': BT})
summary = bt.groupby('Attack_type')['BT'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
summary.to_csv('results/behavioural_trust_by_class.csv')
print('Model used for BT:', best); print(summary)

theta = 0.6
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(BT[yb_te == 0], bins=50, alpha=.7, label='Normal')
ax.hist(BT[yb_te == 1], bins=50, alpha=.7, label='Attack')
ax.axvline(theta, ls='--', color='k', label=f'θ = {theta}')
ax.set_xlabel('Behavioural trust BT = 1 − s'); ax.set_ylabel('Flows'); ax.set_yscale('log'); ax.legend()
fig.tight_layout(); fig.savefig('results/bt_distribution.png', dpi=200); plt.close(fig)
gate = dict(model=best, theta=theta,
            normal_admitted=float((BT[yb_te == 0] >= theta).mean()),
            attack_blocked=float((BT[yb_te == 1] < theta).mean()))
json.dump(gate, open('results/trust_gate.json', 'w'), indent=2); print(gate)

## 10. Save everything and download

In [ ]:
res = pd.DataFrame(ROWS).round(4)
res.to_csv('results/ids_results.csv', index=False)
json.dump(dict(sample_rows=int(len(df)), top_k=TOP_K, features=feat, classes=MUL_LABELS),
          open('results/run_info.json', 'w'), indent=2)
print(res.to_string(index=False))
with zipfile.ZipFile('TAPS_IDS_results.zip', 'w') as z:
    for f in glob.glob('results/*'): z.write(f)
try:
    from google.colab import files; files.download('TAPS_IDS_results.zip')
except Exception:
    print('Download TAPS_IDS_results.zip from the Files panel on the left.')